In [ ]:
import numpy as np
import numpy.lib.recfunctions as recfun
from pathlib import Path
from plyfile import PlyData, PlyElement
import matplotlib.pyplot as plt

In [ ]:
# We start with loading the .ssm data in this cell to keep the data loaded in the following
def load_full_dataset(dataset='Train'):
    # 1) Path Definition
    # base directory for the chosen dataset
    base_path = Path('../data/Challenge-ABC') / dataset
    if not base_path.exists(): # verification
        print(f"Error: Dataset folder {base_path} not found, check 'dataset' argument.")
        return None
    ssm_dir = base_path / 'SSM_Challenge-ABC'
    lb_dir = base_path / 'lb'

    X_list = []
    y_list = []

    ssm_files = list(ssm_dir.glob('*.ssm'))
    total_files = len(ssm_files)

    for i, ssm_file in enumerate(ssm_files, 1):
        file_id = ssm_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 320)
        labels = np.loadtxt(lb_file, dtype='int')

        if len(labels) == len(features):
            X_list.append(features)
            y_list.append(labels)
        else:
            print(f"Error: Dimension error for {file_id} -> ignored.")

        if i % 20 == 0:
            print(f"Loading : {i}/{total_files} files...")

    X = np.vstack(X_list)
    y = np.concatenate(y_list)    
    
    return X, y


X_train, y_train, X_val, y_val = None, None, None, None

# save the datasets in a .npz file for faster loading in the future
npz_file = Path("../data/Challenge-ABC/datasets.npz")
if npz_file.exists():
    print("Loading datasets from .npz file...")
    data = np.load(npz_file)
    X_train, y_train = data['X_train'], data['y_train']
    X_val, y_val = data['X_val'], data['y_val']
else:
    print("Loading datasets from .ssm and .lb files...")
    print("Train dataset loading...")
    X_train, y_train = load_full_dataset('Train')
    print("\nValidation dataset loading...")
    X_val, y_val = load_full_dataset('Validation')
    np.savez(npz_file, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val)
    print("npz file saved for future use.")

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

Loading datasets from .ssm and .lb files...
Train dataset loading...
Loading : 20/198 files...
Loading : 40/198 files...
Loading : 60/198 files...
Loading : 80/198 files...
Loading : 100/198 files...
Loading : 120/198 files...
Loading : 140/198 files...
Loading : 160/198 files...
Loading : 180/198 files...

Validation dataset loading...
Loading : 20/50 files...
Loading : 40/50 files...
npz file saved for future use.
X_train shape: (3174768, 320)
y_train shape: (3174768,)
X_val shape: (690211, 320)
y_val shape: (690211,)


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 64), 
    activation='logistic',            
    solver='adam',                
    max_iter=500,                 
    random_state=42               
)

mlp.fit(X_train, y_train)

y_pred = mlp.predict(X_val)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    650122
           1       0.99      0.96      0.97     40089

    accuracy                           1.00    690211
   macro avg       0.99      0.98      0.99    690211
weighted avg       1.00      1.00      1.00    690211



              precision    recall  f1-score   support

           0       1.00      1.00      1.00    650122
           1       0.99      0.96      0.97     40089

    accuracy                           1.00    690211
   macro avg       0.99      0.98      0.99    690211
weighted avg       1.00      1.00      1.00    690211